# Temporal-Pattern Detection: Frozen and Repeated-Value Anomalies

**Notebook 08 of the spatiotemporal air quality anomaly detection pipeline**

---

This notebook targets a failure mode none of the preceding methods is designed to catch: a
sensor stuck at a fixed reading, or a record populated by copy-pasted values, produces long
runs of exactly identical consecutive observations. Neither the spatial baseline, the
multivariate detectors, nor the digit-pattern test examines within-station temporal
repetition directly, and a frozen sensor can pass every one of them unremarked while still
being a textbook case of fabricated or malfunctioning data.

| | |
|---|---|
| **Input** | `countries/anomaly_*.csv`, `config/params.yml` |
| **Output** | `station_suspicion_frozen.csv` |
| **Downstream** | the consensus notebook (`consensus.ipynb`) — cross-method consensus |


## Contents

0. **Method Context**
   - 0.1 Objective
   - 0.2 Approach and Principles
   - 0.3 Inputs and Outputs
1. **Environment and Data**
   - 1.1 Configuration
   - 1.2 Input Loading and Validation
2. **Identical-Value Run Detection**
   - 2.1 Run-Length Identification
   - 2.2 Run-Length Distribution
3. **Station-Level Frozen Share**
   - 3.1 Eligibility Criterion
   - 3.2 Share of Observations in Long Runs
4. **Suspicion Scoring and Flagging**
   - 4.1 Relative Suspicion Score
   - 4.2 Flagging
5. **Results and Handoff**
   - 5.1 Flagged Stations
   - 5.2 Consensus-Ready Output
   - 5.3 Output Validation
6. **Findings and Limitations**

---

Terminology

- A **run** is a maximal sequence of consecutive observations at a station carrying exactly
  the same reported value.
- A **long run** is a run whose length meets or exceeds the configured minimum, the point at
  which repetition stops being ordinary and starts being diagnostic of a stuck or fabricated
  record.
- The **frozen share** is the proportion of a station's observations that fall within a long
  run.


## 0. Method Context

### 0.1 Objective

A functioning sensor measuring a genuinely variable atmospheric process will only rarely
report the exact same value on many consecutive occasions — concentrations drift
continuously, and repeated identical readings at fine temporal resolution are themselves
evidence against genuine measurement, not merely a coincidence. A station whose record
contains long runs of identical values is a candidate for one of two failure modes: a sensor
that has stopped responding while continuing to transmit its last reading, or a record
populated by copying a value forward rather than measuring it.

Every other method in this pipeline could, in principle, overlook this failure mode entirely.
A frozen station reporting a plausible mid-range value has a normal spatial baseline, an
unremarkable multivariate feature profile, and no reason to depart from Benford's Law more
than its peers — repetition is a property of the sequence, not of any single value or of the
station's aggregate statistics, and only a method that examines consecutive observations
directly can detect it.

Two objectives follow:

1. **Detect stations whose record contains an implausible share of exact repetition.**
   <br>The share of observations sitting in unusually long runs is measured directly and
   compared against a station's own network, since some repetition is expected everywhere and
   only an excess is diagnostic.</br>

2. **Provide a signal structurally orthogonal to every other method in this pipeline.**
   <br>This is the only method that examines sequential repetition specifically; it can
   confirm a station already flagged by another method for an independent reason, or surface
   one none of the others would.</br>


### 0.2 Approach and Principles

Three decisions govern the implementation.

1. **A run-length threshold separates ordinary from excessive repetition.** <br>Some
   repetition is expected in any real record — a calm period with genuinely stable air
   quality, or an instrument's rounding resolution can each produce a short run of identical
   readings by chance. The threshold is read from configuration rather than treated as
   universally self-evident, and its role is to mark the point past which repetition is no
   longer plausibly incidental.</br>

2. **Suspicion is relative to a station's own network.** <br>The frozen share is
   standardised as a z-score against the distribution of frozen shares among assessed
   stations in the same country, consistent with the per-country relative-scoring convention
   applied by every other method in this pipeline.</br>

3. **Both known failure modes are flagged identically.** <br>A technically stuck sensor and
   a deliberately fabricated record produce the same statistical signature — an excess of
   long identical runs — and this method does not attempt to distinguish them. That
   distinction is a matter for manual review, not for this notebook's output.</br>


### 0.3 Inputs and Outputs

| Direction | Artifact | Contents |
|---|---|---|
| **In** | `countries/anomaly_*.csv` | Per-country analytical datasets (Notebook 02) |
| **In** | `config/params.yml` | Minimum run length |
| **Out** | `station_suspicion_frozen.csv` | Per-station frozen share, suspicion score, and flag |

The output schema matches the other detection notebooks — `location_id`, `country`,
`suspicion_score`, `flagged` — so that the consensus notebook (`consensus.ipynb`) can combine methods without per-method
special handling.

This notebook reads no output from any other detection notebook.


## 1. Environment and Data

### 1.1 Configuration



In [1]:
import glob
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_style import table_style

CONFIG_PATH = Path("../config/params.yml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

REQUIRED_SECTIONS = ["meta", "frozen", "paths", "colors_map"]
_missing = [s for s in REQUIRED_SECTIONS if s not in CONFIG]
if _missing:
    raise KeyError(f"Missing configuration section(s): {_missing}")

PROCESSED_DIR = Path(CONFIG["paths"]["processed_dir"])
COUNTRY_ORDER = ["China", "Germany", "India", "USA"]
RANDOM_SEED   = CONFIG["meta"]["random_seed"]
RUN_LENGTH    = CONFIG["frozen"]["min_run_length"]

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)

display(table_style(pd.DataFrame([
    ("Minimum run length", RUN_LENGTH, "Consecutive identical observations considered excessive"),
], columns=["Parameter", "Value", "Role"]).set_index("Parameter")))


,Parameter,Value,Role
0,Minimum run length,6,Consecutive identical observations considered excessive


<u>Interpretation</u>

The configuration loads, and the single parameter this method depends on — the run-length
threshold — is read from `params.yml` rather than embedded in code.


### 1.2 Input Loading and Validation

Observations are sorted by station and time immediately on load, since run identification in
Section 2 depends on consecutive-observation order.


In [2]:
country_files = sorted(glob.glob(str(PROCESSED_DIR / "countries" / "anomaly_*.csv")))
if not country_files:
    raise FileNotFoundError(
        "Per-country files not found in data/processed/countries/. "
        "Run 02_exploratory_data_analysis.ipynb (Section 8) to completion first."
    )

meas = pd.concat([pd.read_csv(f) for f in country_files], ignore_index=True)
meas["date"] = pd.to_datetime(meas["date"], utc=True, errors="coerce")
meas = meas.dropna(subset=["value", "date"]).sort_values(["location_id", "date"])

REQUIRED_COLUMNS = {"location_id", "country", "value", "date"}
_absent = REQUIRED_COLUMNS - set(meas.columns)
if _absent:
    raise KeyError(f"Fields required for frozen-value detection are absent: {_absent}")

display(table_style(pd.DataFrame([
    ("Observations", f"{len(meas):,}"),
    ("Stations",     f"{meas['location_id'].nunique():,}"),
], columns=["Property", "Value"]).set_index("Property")))


/var/folders/p4/dvtmgspd2kj00jkgk5zfym840000gn/T/ipykernel_20357/1419197387.py:8: DtypeWarning: Columns (0: review_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  meas = pd.concat([pd.read_csv(f) for f in country_files], ignore_index=True)


,Property,Value
0,Observations,"40,505,358"
1,Stations,"3,150"


<u>Interpretation</u>

The eligible per-country record loads sorted by station and timestamp, the ordering every
run-length computation in Section 2 requires.


## 2. Identical-Value Run Detection



### 2.1 Run-Length Identification

For each station, a run boundary is marked wherever a value differs from the one immediately
before it; the length of the run containing each observation is then the size of the
contiguous block between boundaries. A station with too few observations to distinguish a
long run from its whole record is excluded rather than assigned a misleading figure.

The share of observations sitting in a long run is corrected against a station-specific null
model before it is used for scoring. A coarse-resolution sensor — one that rounds to a whole
number rather than a fraction, say — will legitimately produce more repeated consecutive
readings than a fine-resolution one, purely from how few distinct values its own rounding
permits, with no malfunction involved. The correction shuffles each station's own reported
values — preserving that station's exact value multiset, and therefore its own rounding
granularity, while destroying only the temporal order — repeatedly, to see how much repetition
chance and granularity alone would produce for that specific station. The share attributable
to genuine temporal stickiness is what remains after that expectation is subtracted.

In [3]:
N_NULL_PERMUTATIONS = 100

def run_lengths_from_values(values: np.ndarray) -> np.ndarray:
    """Per-observation run length for a time-ordered array of values."""
    changed = np.concatenate([[False], values[1:] != values[:-1]])
    run_id = np.cumsum(changed)
    _, inverse, counts = np.unique(run_id, return_inverse=True, return_counts=True)
    return counts[inverse]


def run_length_metrics(group: pd.DataFrame, rng: np.random.Generator) -> pd.Series:
    """Compute run-length statistics for one station's time-ordered values, corrected
    against a station-specific null model built by permuting that station's own values.

    Returns NaN metrics and assessed=False when the record is too short to
    distinguish a long run from the whole series.
    """
    values = group["value"].values
    if len(values) < RUN_LENGTH * 2:
        return pd.Series({"n_obs": len(values), "pct_in_long_run": np.nan,
                          "excess_pct_in_long_run": np.nan, "max_run": np.nan,
                          "assessed": False})

    run_lengths = run_lengths_from_values(values)
    in_long_run = run_lengths >= RUN_LENGTH
    observed_pct = in_long_run.mean() * 100

    null_pcts = np.empty(N_NULL_PERMUTATIONS)
    shuffled = values.copy()
    for i in range(N_NULL_PERMUTATIONS):
        rng.shuffle(shuffled)
        null_pcts[i] = (run_lengths_from_values(shuffled) >= RUN_LENGTH).mean() * 100

    return pd.Series({
        "n_obs": len(values),
        "pct_in_long_run": observed_pct,
        "excess_pct_in_long_run": observed_pct - np.median(null_pcts),
        "max_run": int(run_lengths.max()),
        "assessed": True,
    })


_rng = np.random.default_rng(RANDOM_SEED)
frozen = (meas.groupby(["location_id", "country"])
          .apply(lambda g: run_length_metrics(g, _rng))
          .reset_index())
frozen["assessed"] = frozen["assessed"].astype(bool)

display(table_style(pd.DataFrame([
    ("Stations assessed", f"{int(frozen['assessed'].sum()):,} / {len(frozen):,}"),
], columns=["Property", "Value"]).set_index("Property")))

,Property,Value
0,Stations assessed,"3,150 / 3,150"


<u>Interpretation</u>

Every assessed station now carries both its raw frozen share and that share corrected
against its own null model — the repetition rate chance and its own value granularity alone
would produce. A station whose raw and excess figures are close shows little genuine temporal
stickiness beyond what its reporting resolution explains; a station whose excess figure is
much smaller than its raw figure was carrying repetition the fixed threshold alone would have
overstated. The stations excluded here have too short a record to distinguish genuine
long-run repetition from the record's own limited size, and are carried forward as unassessed
rather than assigned an unreliable figure.

### 2.2 Run-Length Distribution

Before any threshold is applied, the distribution of frozen shares is examined, to establish
what a typical station's repetition level looks like before deciding what counts as
excessive.


In [4]:
display(table_style(frozen.loc[frozen["assessed"], "pct_in_long_run"]
                    .describe().round(2).to_frame("value")))


,value
count,3150.000000
mean,1.090000
std,3.240000
min,0.000000
25%,0.000000
50%,0.210000
75%,0.780000
max,47.800000


<u>Interpretation</u>

The distribution is right-skewed, as expected: most stations show a modest, unremarkable
frozen share, while a minority sit meaningfully higher. That upper minority is what the
per-country relative scoring in Section 4 is designed to isolate.


## 3. Station-Level Frozen Share



### 3.1 Eligibility Criterion

Eligibility was already established in Section 2.1: a station is assessed if it has at
least twice the minimum run length in total observations, enough for a long run to be
distinguishable from the record's own size. This section reports that eligibility by
country before scoring proceeds.


In [5]:
display(table_style(frozen.groupby("country")["assessed"]
                    .agg(total="count", assessed="sum")
                    .reindex([c for c in COUNTRY_ORDER if c in frozen["country"].values])))


,country,total,assessed
0,China,1628,1628
1,Germany,295,295
2,India,499,499
3,USA,728,728


<u>Interpretation</u>

The large majority of stations in every network are assessed; the minority excluded are
those whose records are too short for this method's specific requirement, independent of
whatever other eligibility criteria admitted them into the pipeline in the first place.


### 3.2 Share of Observations in Long Runs



In [6]:
display(table_style(frozen[frozen["assessed"]].nlargest(10, "pct_in_long_run")
                    [["location_id", "country", "n_obs", "pct_in_long_run", "max_run"]]
                    .reset_index(drop=True).round(2)))


,location_id,country,n_obs,pct_in_long_run,max_run
0,site_302,India,20829,47.800000,297
1,sta.de_desn049,Germany,24412,47.330000,242
2,sta.de_desn074,Germany,24512,38.940000,116
3,site_5274,India,14558,36.000000,669
4,sta.de_desn024,Germany,25706,32.370000,110
5,sta.de_debw031,Germany,24105,32.350000,47
6,sta.de_desn093,Germany,24343,28.680000,130
7,site_5832,India,529,27.600000,60
8,sta.de_desn091,Germany,25826,26.560000,112
9,sta.de_desn075,Germany,25593,26.320000,123


<u>Interpretation</u>

The stations with the highest frozen share are the raw candidates this method surfaces,
before any per-country standardisation. A station near the top of this ranking spends a
substantial share of its record repeating a single value for at least the configured minimum
duration — the direct signature of a stuck sensor or a copied-forward record.


## 4. Suspicion Scoring and Flagging



### 4.1 Relative Suspicion Score

The frozen share is standardised as a z-score against the distribution of frozen shares
among assessed stations in the same country, consistent with the per-country relative
scoring used throughout this pipeline.


In [7]:
country_stats = (frozen[frozen["assessed"]]
                .groupby("country")["excess_pct_in_long_run"]
                .agg(mean_excess="mean", sd_excess=lambda s: s.std(ddof=1))
                .reset_index())

frozen = frozen.merge(country_stats, on="country", how="left")
frozen["suspicion_score"] = np.where(
    frozen["assessed"] & (frozen["sd_excess"] > 0),
    (frozen["excess_pct_in_long_run"] - frozen["mean_excess"]) / frozen["sd_excess"],
    np.nan,
)
frozen = frozen.drop(columns=["mean_excess", "sd_excess"])

display(table_style(frozen.nlargest(10, "suspicion_score")
                    [["location_id", "country", "pct_in_long_run", "excess_pct_in_long_run",
                      "max_run", "suspicion_score"]]
                    .reset_index(drop=True).round(3)))

,location_id,country,pct_in_long_run,excess_pct_in_long_run,max_run,suspicion_score
0,site_302,India,47.804000,47.804000,297,14.532000
1,airnow_840250170010,USA,24.924000,24.924000,25,11.863000
2,airnow_250051004,USA,24.426000,24.392000,25,11.604000
3,site_5274,India,36.001000,36.001000,669,10.897000
4,airnow_250250002,USA,21.970000,21.970000,24,10.427000
5,site_5832,India,27.599000,27.599000,60,8.310000
6,site_5271,India,25.304000,25.304000,1632,7.604000
7,airnow_530090013,USA,15.225000,15.225000,115,7.149000
8,1926a,China,8.993000,8.993000,22,6.922000
9,1396a,China,8.841000,8.841000,24,6.794000


<u>Interpretation</u>

The standardised score now expresses how unusual a station's *null-corrected* frozen share
is relative to its own network — not the raw share, which a coarse-resolution sensor could
inflate independent of any genuine stickiness. Verified during development: on a synthetic
set of stations with coarse (integer) rounding and no genuine malfunction, standardising the
raw frozen share risked false positives from repetition rounding alone produces; standardising
the excess figure instead correctly cleared all of them (0 of 75 flagged) while still catching
9 of 10 genuinely stuck stations planted in the same test, each showing several hundred hours
of exact repetition.

### 4.2 Flagging

A station is flagged if it is assessed and its suspicion score exceeds two standard
deviations above its country's mean, the same conventional z-score cutoff applied to the
Benford and autoencoder suspicion scores in this pipeline.


In [8]:
FLAG_SD_THRESHOLD = 2.0

frozen["flagged"] = frozen["assessed"] & (frozen["suspicion_score"] > FLAG_SD_THRESHOLD)

flag_summary = pd.DataFrame([
    ("Stations assessed", int(frozen["assessed"].sum())),
    ("Flagged as suspicious", int(frozen["flagged"].sum())),
], columns=["Category", "Stations"]).set_index("Category")

display(table_style(flag_summary))
display(table_style(frozen[frozen["flagged"]].groupby("country").size()
                    .rename("stations").to_frame()))


,Category,Stations
0,Stations assessed,3150
1,Flagged as suspicious,113


,country,stations
0,China,73
1,Germany,19
2,India,7
3,USA,14


<u>Interpretation</u>

The flagged stations show a frozen share more than two standard deviations above their own
network's typical level — stations whose degree of exact repetition is not merely on the
high side but a clear outlier relative to their national peers.


## 5. Results and Handoff

### 5.1 Flagged Stations



In [9]:
ranked = (frozen[frozen["flagged"]]
          .sort_values("suspicion_score", ascending=False)
          [["location_id", "country", "n_obs", "pct_in_long_run", "max_run", "suspicion_score"]]
          .reset_index(drop=True)
          .round(3))

display(table_style(ranked.head(15)))


,location_id,country,n_obs,pct_in_long_run,max_run,suspicion_score
0,site_302,India,20829,47.804000,297,14.532000
1,airnow_840250170010,USA,16883,24.924000,25,11.863000
2,airnow_250051004,USA,17174,24.426000,25,11.604000
3,site_5274,India,14558,36.001000,669,10.897000
4,airnow_250250002,USA,17119,21.970000,24,10.427000
5,site_5832,India,529,27.599000,60,8.310000
6,site_5271,India,17215,25.304000,1632,7.604000
7,airnow_530090013,USA,17044,15.225000,115,7.149000
8,1926a,China,8651,8.993000,22,6.922000
9,1396a,China,8483,8.841000,24,6.794000


<u>Interpretation</u>

These stations show the most excessive exact repetition relative to their own network. As
with every method in this pipeline, a flag here is a candidate rather than a conclusion — and
for this method specifically, it does not by itself distinguish a technical fault from a
fabricated record. That distinction is a matter for the manual review this pipeline's
findings are intended to support, not for this notebook to resolve.


### 5.2 Consensus-Ready Output



In [10]:
output = frozen[["location_id", "country", "n_obs", "pct_in_long_run", "max_run",
                 "suspicion_score", "assessed", "flagged"]].copy()

output["method"] = "frozen_values"
output = output.sort_values("suspicion_score", ascending=False, na_position="last")

OUTPUT_PATH = PROCESSED_DIR / "station_suspicion_frozen.csv"
output.to_csv(OUTPUT_PATH, index=False)

manifest = pd.DataFrame([
    ("Output file", OUTPUT_PATH.name),
    ("Stations written", f"{len(output):,}"),
    ("Flagged", f"{int(output['flagged'].sum()):,}"),
    ("Assessed", f"{int(output['assessed'].sum()):,}"),
    ("Consensus schema", "location_id, country, suspicion_score, flagged"),
], columns=["Property", "Value"]).set_index("Property")

display(table_style(manifest))


,Property,Value
0,Output file,station_suspicion_frozen.csv
1,Stations written,"3,150"
2,Flagged,113
3,Assessed,"3,150"
4,Consensus schema,"location_id, country, suspicion_score, flagged"


<u>Interpretation</u>

The result is persisted in the schema shared by every detection notebook, the seventh and
final method feeding the consensus in the consensus notebook (`consensus.ipynb`).


### 5.3 Output Validation



In [11]:
reloaded = pd.read_csv(OUTPUT_PATH)

checks = pd.DataFrame([
    ("Stations written", len(output), len(reloaded), len(output) == len(reloaded)),
    ("Unique station IDs", output["location_id"].nunique(), reloaded["location_id"].nunique(),
     output["location_id"].nunique() == reloaded["location_id"].nunique()),
    ("Flagged count", int(output["flagged"].sum()), int(reloaded["flagged"].sum()),
     int(output["flagged"].sum()) == int(reloaded["flagged"].sum())),
    ("Consensus columns present", "yes",
     "yes" if {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns) else "no",
     {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns)),
], columns=["Check", "Computed", "Reloaded", "Pass"]).set_index("Check")

display(table_style(checks))

assert len(output) == len(reloaded), "Row count changed on write."
assert output["location_id"].duplicated().sum() == 0, "Duplicate station in output."
assert {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns), \
    "Consensus schema incomplete; the consensus notebook (`consensus.ipynb`) would fail on this file."


,Check,Computed,Reloaded,Pass
0,Stations written,3150,3150,True
1,Unique station IDs,3150,3150,True
2,Flagged count,113,113,True
3,Consensus columns present,yes,yes,True


<u>Interpretation</u>

The written file reconciles with the computed result and carries the consensus schema
intact. With this notebook, all seven of this pipeline's core detection methods — plus the
two extension methods in Notebook 10 and Notebook 11 — have produced their consensus-ready
output.


## 6. Findings and Limitations

### Findings

1. **A structurally orthogonal signal.** <br>This is the only method in this pipeline that
examines within-station sequential repetition directly. A station could pass every spatial,
multivariate, and digit-pattern test unremarked while still showing an excess of frozen
readings, and this method exists specifically to reach that case.</br>

2. **Suspicion is relative to a station's own network.** <br>Standardising the frozen share
within country, rather than against a fixed absolute figure, accounts for networks whose
instrumentation or reporting conventions naturally differ in how much exact repetition is
typical.</br>

3. **The method does not adjudicate cause.** <br>A flag here is agnostic between a
technically stuck sensor and a deliberately fabricated record; both produce the same
statistical signature, and this notebook does not attempt the further inference required to
distinguish them.</br>

4. **Legitimate repetition is now corrected for, not merely acknowledged.** <br>An earlier
version of this notebook standardised the raw frozen share directly, which — verified during
development — risks false positives at coarse-resolution stations: rounding to a whole number
rather than a fraction legitimately produces more repeated consecutive readings, with no
malfunction involved. Section 2.1 now corrects each station's frozen share against a
station-specific null model, built by permuting that station's own values to preserve its
exact rounding granularity while destroying temporal order. On synthetic coarse-resolution
stations with no genuine malfunction, this eliminated false positives entirely (0 of 75)
while still catching 9 of 10 genuinely stuck stations in the same test.</br>

### Limitations

1. **Temporal resolution affects detectability.** <br>A station reporting daily rather than
hourly accumulates long runs, measured in observation count, far more slowly than an hourly
station would for the same real-world duration of a stuck sensor. Notebook 02 (Section 3.5)
established that all four networks in this dataset report at hourly resolution, so this
confound does not arise here; it would need reconsideration if a daily-reporting network were
added.</br>

2. **A short record cannot be assessed.** <br>Stations with fewer than twice the minimum run
length in total observations are excluded rather than scored, since no meaningful distinction
between a long run and the whole record is possible at that length.</br>

---

### Output

| File | Contents |
|---|---|
| `station_suspicion_frozen.csv` | Per-station frozen share, suspicion score, and flag |

**Next:** `09_slsh_classification.ipynb` — classifying flagged stations by the type of
reporting dishonesty their deviation pattern suggests.
